# Feature Analysis
- Umap(Visualization embedding space)
- box plot(Metastasis vs. Primary)
- correlation(similarity score and pubchem features)

In [ ]:
import pandas as pd
from pyteomics import mgf
import re
import numpy as np
import json
import pickle    
from scipy.spatial.distance import pdist, squareform


### 1. Filter by MS2 identifier
- filter the MS1 features by their appearance in the MS2 scan


In [ ]:
mz_dataset = json.load(open("../../data/medical/RFA MSMS_mz_precursor.json"))
n_dataset = json.load(open("../../data/medical/RFA MSMS_n.json"))
msms_dataset = mz_dataset + n_dataset

In [ ]:
sample_id = ...
features = pd.read_csv("../../data/medical/ST003752_AN006162_Results.txt", sep="\t")
select_cols = ['Compound','S_123', 'S_124', 'S_125', 'S_126', 'S_127', 'S_128', 'S_129', 'S_130', 'S_131', 'S_132', 'S_133', 'S_134', "S_135", "S_136", "S_137", "S_138", "S_139"]
features = features[select_cols]
print(features.head())

In [ ]:
#count the number of features where all samples have a value of 0
num_zero_features = (features.iloc[:, 1:] == 0).all(axis=1).sum()
print(f"Number of features where all samples have a value of 0: {num_zero_features}")
#example of a feature where all samples have a value of 0
zero_feature = features[(features.iloc[:, 1:] == 0).all(axis=1)]
print(zero_feature)
#drop features where all samples have a value of 0
features = features[~(features.iloc[:, 1:] == 0).all(axis=1)]
print(f"Number of features after dropping features where all samples have a value of 0: {len(features)}")

In [ ]:

# dictionary of MS2 by ID
msms_dict = {entry['identifier']: entry for entry in msms_dataset}
print(f"Number of unique MS/MS spectra: {len(msms_dict)}")
print(f"First 10 MS/MS spectrum IDs: {list(msms_dict.keys())[:10]}")

In [ ]:
counter = 0
for inex, feature in features.iterrows():
    feature_id = feature['Compound']
    if feature_id.endswith("m/z"):
        print(f"Feature {feature_id} has a matching MS/MS spectrum")
        counter += 1
    if counter >= 10:
        break
test = '1.50_244.0692n' #'0.00_141.0026m/z'
if test in msms_dict:
    print(f"Test feature {test} has a matching MS/MS spectrum")

In [ ]:
# matching MS1 features to MS2 spectra
features_filtered = features.copy()
counter = 0
for inex, feature in features_filtered.iterrows():
    feature_id = feature['Compound']
    #print(f"Checking feature {feature_id} against MS/MS dataset")
    if feature_id not in msms_dict:
        #drop row from dataset
        
        features_filtered.drop(inex, inplace=True)
    else:
        counter += 1
print(f"Number of features after filtering for MS/MS spectra: {counter} == {len(features_filtered)}")

## 4. Most significant Features
- Volcano plot

In [ ]:
# import IDs from metadata
ids = pd.read_csv("../../data/medical/Sample_ID.csv", index_col=0)
ids

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
from scipy.stats import ttest_ind

def plot_volcano(
    merged_df,
    ids,
    classification_col='sample ID',
    sample_prefix='S_',
    group_col='condition',
    group1='Primary',
    group2='Metastatic',
    fc_threshold=1.0,         # log2 fold-change threshold
    p_threshold=0.05,
    agg='mean',               # how to combine feature rows (mean or sum)         
    figsize=(8,6),
    verbose=True
):
    """
    Create a volcano plot comparing abundances between two groups (e.g. Primary vs Metastatic).

    Parameters
    ----------
    merged_df : pd.DataFrame
        DataFrame with abundance columns (e.g. S_123...) and a column for feature identity/classification.
    ids : pd.DataFrame
        DataFrame with sample IDs and group information.
    classification_col : str
        Column identifying features or categories (e.g. 'identifier', 'class', 'pathway').
    sample_prefix : str
        Prefix for sample columns (e.g. 'S_').
    group_col : str
        Column in ids specifying the grouping variable.
    group1, group2 : str
        Names of the two groups to compare.
    fc_threshold : float
        Absolute log2 fold-change cutoff for significance.
    p_threshold : float
        p-value cutoff for significance.
    agg : str
        Aggregation method if multiple rows per classification ('mean' or 'sum').
    figsize : tuple
        Size of the matplotlib figure.
    """

    # Detect sample columns
    sample_cols = [c for c in merged_df.columns if isinstance(c, str) and c.startswith(sample_prefix)]
    merged_df = merged_df.copy()
    merged_df[sample_cols] = merged_df[sample_cols].apply(pd.to_numeric, errors='coerce').fillna(0)

    # Map samples to group
    group_map = ids[group_col].to_dict()
    merged_df = merged_df.loc[:, [classification_col] + sample_cols]

    # Aggregate by classification if needed
    if agg == 'sum':
        agg_df = merged_df.groupby(classification_col)[sample_cols].sum()
    else:
        agg_df = merged_df.groupby(classification_col)[sample_cols].mean()

    # Compute fold-change and p-values
    results = []
    for feature, row in agg_df.iterrows():
        sample_values = row.to_dict()
        group1_vals = [v for s, v in sample_values.items() if group_map.get(s) == group1]
        group2_vals = [v for s, v in sample_values.items() if group_map.get(s) == group2]
        if len(group1_vals) < 2 or len(group2_vals) < 2:
            continue
        # Mann-Whitney U test
        # stat, pval = mannwhitneyu(group1_vals, group2_vals, alternative='two-sided')

        # Use t-test instead

        stat, pval = ttest_ind(group1_vals, group2_vals, equal_var=False)

        # Log2 fold change (Metastatic / Primary)
        fc = np.mean(group2_vals) / (np.mean(group1_vals) + 1e-12)
        log2fc = np.log2(fc)
        results.append({'Feature': feature, 'log2FC': log2fc, 'pval': pval})

    res_df = pd.DataFrame(results).dropna()
    res_df['neglog10p'] = -np.log10(res_df['pval'])

    # Determine significance
    res_df['Significance'] = 'Not significant'
    res_df.loc[(res_df['pval'] < p_threshold) & (res_df['log2FC'] > fc_threshold), 'Significance'] = 'Increased in ' + group2
    res_df.loc[(res_df['pval'] < p_threshold) & (res_df['log2FC'] < -fc_threshold), 'Significance'] = 'Decreased in ' + group2

    # Plot
    plt.figure(figsize=figsize)
    sns.set_theme(style="whitegrid", context="talk")

    sns.scatterplot(
        data=res_df,
        x='log2FC',
        y='neglog10p',
        hue='Significance',
        palette={'Increased in ' + group2: 'red', 'Decreased in ' + group2: 'blue', 'Not significant': 'gray'},
        alpha=0.7,
        s=70
    )

    # Threshold lines
    plt.axvline(fc_threshold, color='black', linestyle='--', linewidth=1)
    plt.axvline(-fc_threshold, color='black', linestyle='--', linewidth=1)
    plt.axhline(-np.log10(p_threshold), color='black', linestyle='--', linewidth=1)

    # plt.title(f"Volcano plot: {group2} vs {group1}")
    plt.xlabel(f"log2(Fold Change) ({group2}/{group1})")
    plt.ylabel("-log10(p-value)")
    plt.legend(title='')
    plt.tight_layout()
    plt.show()



    if verbose:
        sig_counts = res_df['Significance'].value_counts()
        print("\nSignificant features summary:")
        print(sig_counts.to_string())

    return res_df

In [ ]:
volcano_df = plot_volcano(
    features_filtered,
    ids,
    classification_col='Compound',   # or 'class', 'pathway'
    #group1='Metastasis (group A)',
    #group2='Metastasis (group B)',
    #group_col = 'condition',
    group1='Primary',
    group2='Metastasis',
    group_col='condition',
    fc_threshold=1.0,
    p_threshold=0.05,
    figsize=(8,6)
)

In [ ]:
# Select 50 most significant features (by p-value) for further analysis that are also significant
top_features = volcano_df.sort_values('pval')
top_features.columns
volcanofeatures = top_features[top_features['Significance'] != 'Not significant']
top_features = volcanofeatures.head(50)

print("\nTop 50 significant features:")
top_features
# Columns: Feature, log2FC, pval, neglog10p, Significance

## 5. Annotate Features
- using pubchem lite and Jestr

In [ ]:
# Display results
with open("../../experiments/20260716_PUBCHEM_sample_run_3/result_RFA_MSMS_full_precursor.pkl", "rb") as f:

    results = pickle.load(f)
#len(results)
#results.columns
results.head()

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

# --- Data collection ---
similarity_scores = []
ppm_errors = []
for row in results.itertuples():
    if not row.scores:
        raise ValueError(f"No candidates for feature {row.identifier}")
    similarity_scores.append(max(row.scores))
    ppm_errors.append(row.ppm_error[0] if row.ppm_error else np.nan)

# --- Convert to arrays, drop NaN ---
scores = np.array(similarity_scores)
errors = np.array(ppm_errors, dtype=float)
valid_mask = ~np.isnan(errors)
scores, errors = scores[valid_mask], errors[valid_mask]

# --- Normalize ---
def normalize(arr):
    if len(arr) == 0: return arr
    arr = arr - min(arr) if min(arr) < 0 else arr
    return arr / max(arr) if max(arr) > 0 else arr

scores_norm = normalize(scores)
errors_norm = normalize(errors)

# --- Plot with density visualization ---
correlation = pd.DataFrame({'similarity_scores': scores_norm, 'ppm_errors': errors_norm})
raw_corr = correlation.corr(method='pearson').iloc[0, 1]

plt.figure(figsize=(12, 4))
plt.figure(figsize=(10, 6))
plt.yscale('logit')
sns.regplot(data=correlation, x='similarity_scores', y='ppm_errors',
            scatter_kws={'alpha':0.3, 's':10},  # Optional: keep faint points
            line_kws={'color':'red', 'linewidth':2},
            ci=95)  # Confidence interval
plt.title('Linear Regression with Confidence Band')
plt.show()


# --- Pearson r and p-value ---
r, p_value = pearsonr(correlation['similarity_scores'], correlation['ppm_errors'])
print(f"Pearson r: {r:.4f}, p-value: {p_value:.4e}")

In [ ]:
#Filter for ppm <3ppm
filtered_results = results.copy()
for row in filtered_results.itertuples():
    if row.ppm_error[0] >= 3:
        filtered_results.drop(row.Index, inplace=True)
print(f"Number of results with ppm error < 3: {len(filtered_results)}")

In [ ]:
# Select the highest scoring candidate for each feature
unified_features = {}
smiles_total = set()
for row in filtered_results.itertuples():
    identifier = re.sub(r"_[^_]*$", "", row.identifier)
    candidates_list = row.candidates
    score_list = row.scores
    combined = list(zip(candidates_list, score_list))
    best = max(combined, key=lambda x: x[1])
    if unified_features.get(identifier) and unified_features[identifier]['score'] > best[1]:
        continue
    #unpack embedding from row
    embedding = row.embedding[0][0]
    if best[1] >= 0.35: 
        confidence_group = 'high'
    else: 
        confidence_group = 'low'
    unified_features[identifier] = {'candidate': best[0], 'score': best[1], 'embeddings': embedding, 'confidence_group': confidence_group}
    smiles_total.add(best[0])
print(f"Number of unique features after unification: {len(unified_features)}")
print(f"Number of unique SMILES after unification: {len(smiles_total)}")
# test 
test_id = '0.57_348.9001n'
print(f"First 5 unified features: {list(unified_features.items())[:5]}")
print(f"Test feature {test_id}: {unified_features.get(test_id)}")
# print column from results
for row in results.itertuples():
    if row.identifier.startswith(test_id):
        id = row.identifier
        print(f"Original candidates for {id}: {row.candidates}")
        print(f"Original scores for {id}: {row.scores}")

smiles_total = list(smiles_total)

### Smiles of the most significant features

In [ ]:
#top_features
# Columns: Feature, log2FC, pval, neglog10p, Significance
annotations = []
top_unified_features = {}
for feature in top_features['Feature']:
    if feature in unified_features:
        annotations.append({'feature': feature, 'result': unified_features.get(feature)})
        top_unified_features[feature] = unified_features.get(feature)
print(len(annotations))
smiles = [annotation['result']['candidate'] for annotation in annotations] if annotations else AttributeError("No annotations found for top features")
#unify smiles
smiles = set(smiles)
smiles = list(smiles)
print(len(smiles))

In [ ]:
# Assign classes with NLP
# 30 minutes on 5.9k 
import requests

def NPclassifier_query(smiles):
    endpoint = "https://npclassifier.gnps2.org/classify"
    req_data = {"smiles": smiles}
    out = requests.get(f"{endpoint}", data=req_data)
    out.raise_for_status()
    out_json = out.json()
    return out_json

# smiles_to_class = {}
# for s in smiles_total:
#     #print(f"Processing SMILES: {s}")
#     out_json = NPclassifier_query(s)
#     smiles_to_class[s] = {'class': out_json['class_results'], 'superclass':out_json['superclass_results'], 'pathway': out_json['pathway_results']}
#     if len(smiles_to_class) % 100 == 0:
#         print(f"Processed {len(smiles_to_class)} SMILES")
# #save into data folder
# with open("../../data/medical/smiles_to_class_precursors.json", "w") as f:
#     json.dump(smiles_to_class, f, indent=4)

## 6. Plot UMAP


In [ ]:
# select top 20 superclasses
smiles_to_class_df = json.load(open("../../data/medical/smiles_to_class_precursors.json", "r"))
#create table of all features annotated with classes 
# extend unified features dictionary with class annotation

for identifier, feature_data in unified_features.items():
    candidate = feature_data['candidate']
    if candidate in smiles_to_class_df:
        entry = smiles_to_class_df[candidate]
        if entry['superclass']:
            npclass = entry['superclass'][0]
            unified_features[identifier]['superclass'] = npclass
class_count_high = dict()

class_count_low = dict()

for identifier, feature_data in unified_features.items():
    if 'superclass' in feature_data:
        npclass = feature_data['superclass']
        #low confidence class count
        if feature_data['score'] < 0.7:
            if npclass not in class_count_low:
                class_count_low[npclass] = 1
            else:
                class_count_low[npclass] += 1

        #high confidence class count
        if feature_data['score'] >= 0.7:
            if npclass not in class_count_high:
                class_count_high[npclass] = 1
            else:
                class_count_high[npclass] += 1

        
# sort class count by count
class_count = dict(sorted(class_count_high.items(), key=lambda item: item[1], reverse=True))
# filter for the 20 most frequent superclasses
top_20_classes = list(class_count.keys())[:20]
umap_features = {
    identifier: feature_data
    for identifier, feature_data in unified_features.items()
    if feature_data.get('superclass') in top_20_classes
}
print("Top 20 superclasses:")
average_count = []
for i, (npclass, count) in enumerate(class_count.items()):
    if i >= 20:
        break
    print(f"{npclass}: {count} features")
    average_count.append(count)
print(f" Sample item of dictionary entry: {list(umap_features.items())[0]}")
print(f"Average number of features per superclass: {sum(average_count) / len(average_count):.2f}")
#largest group in low confidence group
if class_count_low:
    largest_low_class = max(class_count_low, key=class_count_low.get)
    largest_low_count = class_count_low[largest_low_class]
    print(f"Largest superclass in low confidence group: {largest_low_class} with {largest_low_count} features")

In [ ]:
#Group 1: features with low confidence <0.35
#Group 2: features with high confidence >=0.35
confidence_count = {'low': 0, 'high': 0}
for identifier, feature_data in umap_features.items():
    if feature_data['score'] < 0.7:
        umap_features[identifier]['confidence_group'] = 'low'
        confidence_count['low'] += 1
    else:
        umap_features[identifier]['confidence_group'] = 'high'
        confidence_count['high'] += 1
print(f"Number of features with low confidence: {confidence_count['low']}")
print(f"Number of features with high confidence: {confidence_count['high']}")
#count classes maximum class count

In [ ]:
import umap
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

SEED = 3

# ---------------------------------------------------------
# 1. Extract embeddings, classes, and confidence flags
# ---------------------------------------------------------

embeddings = []
classes = []
confidence = []

for key, entry in umap_features.items():
    embeddings.append(entry["embeddings"])
    classes.append(entry["superclass"])
    confidence.append(entry["confidence_group"])

embeddings = np.array(embeddings)

# ---------------------------------------------------------
# 2. Fit UMAP
# ---------------------------------------------------------

reducer = umap.UMAP(
    n_neighbors=199, #45
    metric='cosine',
    min_dist=0.8, #0.8
    
    random_state=SEED
)

embeddings_2d = reducer.fit_transform(embeddings)

# ---------------------------------------------------------
# 3. Prepare colors
# ---------------------------------------------------------

unique_classes = sorted(set(classes))
label_to_int = {cls: i for i, cls in enumerate(unique_classes)}
int_labels = np.array([label_to_int[c] for c in classes])

cmap = plt.cm.get_cmap("tab20", len(unique_classes))

# ---------------------------------------------------------
# 4. Plot high-confidence points
# ---------------------------------------------------------

plt.figure(figsize=(10, 7))

high_mask = np.array(confidence) == "high"

plt.scatter(
    embeddings_2d[high_mask, 0],
    embeddings_2d[high_mask, 1],
    c=int_labels[high_mask],
    cmap=cmap,
    s=25,
    alpha=0.9,
    linewidths=0,
    label="High confidence"
)

# ---------------------------------------------------------
# 5. Compute and plot low-confidence centroids
# ---------------------------------------------------------

low_mask = np.array(confidence) == "low"

low_embeddings = embeddings_2d[low_mask]
low_classes = np.array(classes)[low_mask]
low_int_labels = int_labels[low_mask]

# Compute centroids per class
# centroids = {}
# for cls in unique_classes:
#     cls_mask = low_classes == cls
#     if np.sum(cls_mask) > 0:
#         centroids[cls] = low_embeddings[cls_mask].mean(axis=0)


centroids = {}
for cls in unique_classes:
    cls_mask = low_classes == cls
    pts = low_embeddings[cls_mask]
    if len(pts) == 0:
        continue
    # If only one point, that point is the medoid
    if len(pts) == 1:
        centroids[cls] = pts[0]
        continue
    # Pairwise distances
    dist_matrix = squareform(pdist(pts))
    # Sum distances from each point to all others
    total_dist = dist_matrix.sum(axis=1)
    # Index of the point with smallest total distance
    medoid_idx = np.argmin(total_dist)
    centroids[cls] = pts[medoid_idx]



# Plot centroids
for cls, center in centroids.items():
    plt.scatter(
        center[0], center[1],
        color=cmap(label_to_int[cls]),
        s=120,
        edgecolors="black",
        linewidths=1.5,
        alpha=1.0,
        marker="o",
        label=f"{cls} (low conf centroid)"
    )

# ---------------------------------------------------------
# 6. Legend for classes
# ---------------------------------------------------------

class_patches = [
    mpatches.Patch(color=cmap(label_to_int[cls]), label=cls)
    for cls in unique_classes
]

plt.legend(
    handles=class_patches + [
        mpatches.Patch(facecolor="white", edgecolor="black", label="Low confidence centroid")
    ],
    bbox_to_anchor=(1.05, 1),
    loc="upper left",
    fontsize=10
)
plt.xlabel("UMAP1")
plt.ylabel("UMAP2")

plt.xticks([]); plt.yticks([])
plt.box(False)
plt.tight_layout()
plt.show()



## 6. Pubchem lite Patent count and literature features
- can metastatic and primary cancer be differentiated with pubchem count 
- plot box blots with significance 

In [ ]:
#extend dictionary to include pubchemlite annotations
pubchemlite = pd.read_csv("../../data/pubchemlite/PubChemLite_CCSbase_20260529.csv")
pubchemlite.columns
pubchem_feature_dict = dict()
for index, row in pubchemlite.iterrows():
    smiles = row['SMILES']
    pubchem_feature_dict[smiles] = {
        'PubMed_Count': row['PubMed_Count'],
        'Patent_Count': row['Patent_Count'],
        'DrugMedicInfo': row['DrugMedicInfo'],
        'FoodRelated': row['FoodRelated'],
        'PharmacoInfo': row['PharmacoInfo'],
        'SafetyInfo': row['SafetyInfo'],
        'ToxicityInfo': row['ToxicityInfo'],
        'KnownUse': row['KnownUse'],
        'DisorderDisease': row['DisorderDisease']
    }
# Features to be used PubMed_Count, Patent_Count
#  'DrugMedicInfo', 'FoodRelated', 'PharmacoInfo', 'SafetyInfo', 'ToxicityInfo', 'KnownUse', 'DisorderDisease'
skipped = 0
#high_conf_unified_features = {k: v for k, v in top_unified_features.items() if v['confidence_group'] == 'high'}
pubchem_unified_features = unified_features.copy()
for identifier, feature_data in pubchem_unified_features.items():
    candidate = feature_data['candidate']
    # drop all features where confidence is low <0.35
    # if feature_data['score'] < 0.35:
    #     continue
    if candidate in pubchem_feature_dict:
        pubchem_unified_features[identifier]['pubchem'] = pubchem_feature_dict[candidate]
    else:
        skipped += 1
high_conf_unified_features = {k: v for k, v in pubchem_unified_features.items() if v['confidence_group'] == 'high'}
print(f"Number of features with PubChemLite annotations: {len(high_conf_unified_features) - skipped}")

In [ ]:
#filter top features 
topfeature_dict = volcanofeatures.set_index('Feature').to_dict(orient='index')
print(f"Number of top features: {len(topfeature_dict)}")
high_conf_top_features = {}
for identifier, feature_data in high_conf_unified_features.items():
    if identifier in topfeature_dict:
        high_conf_top_features[identifier] = feature_data
print(f"Number of high confidence features that are also top features: {len(high_conf_top_features)}")

In [ ]:
print(f"Sample unified feature entry with PubChemLite annotation: {list(high_conf_unified_features.items())[0]}")
print(f"features_filtered.columns: {features_filtered.columns.tolist()}")
print(f"ids: {ids}")

### Metastasis vs. Primary

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from math import ceil
from scipy.stats import mannwhitneyu
import textwrap
import math

sns.set_theme(style="whitegrid", context="talk")


def plot_pubchem_boxplots(
    unified_features,
    features_filtered,
    ids,
    metadata_col='condition',
    top_n=None,
    min_features=1,
    figsize_per_plot=(5, 6),
    show_plot=True,
    add_stats=True
):
    """
    Create boxplots comparing PubChemLite features between Primary and Metastatic cancers.
    Zeros are excluded from plotting and statistics.
    """
    
    # Get sample columns from features_filtered
    sample_cols = [col for col in features_filtered.columns if col.startswith('S_')]
    
    if not sample_cols:
        raise ValueError("No sample columns (starting with 'S_') found in features_filtered")
    
    # Filter to samples present in ids
    sample_cols = [col for col in sample_cols if col in ids.index]

    #dictionary to track sample size for each feature and contidion key: feature , value: primary_count, metastatic_count
    feature_sample_size = {}
    
    # Prepare plot data
    plot_data = []
    for idx, row in features_filtered.iterrows():
        compound_name = row['Compound']
        
        if compound_name not in unified_features:
            continue
            
        pubchem_data = unified_features[compound_name].get('pubchem', {})
        if not pubchem_data:
            continue
        
        pubchem_numerical = {k: v for k, v in pubchem_data.items() 
                            if isinstance(v, (int, float))}
        
        if not pubchem_numerical:
            continue
        
        for sample_col in sample_cols:
            abundance = row[sample_col]
            sample_condition = ids.loc[sample_col, metadata_col]
            
            for pubchem_feature, pubchem_value in pubchem_numerical.items():
                value = math.log(1+ pubchem_value * abundance)
                if value == 0:
                    continue
                plot_data.append({
                    'Feature': pubchem_feature,
                    'Value': value,
                    'Condition': sample_condition,
                    'Compound': compound_name,
                    'Sample': sample_col
                })
                if feature_sample_size.get(pubchem_feature):
                    feature_sample_size[pubchem_feature][sample_condition] = feature_sample_size[pubchem_feature].get(sample_condition, 0) + 1
                else:
                    feature_sample_size[pubchem_feature] = {sample_condition: 1}
    
    plot_df = pd.DataFrame(plot_data)
    
    if plot_df.empty:
        raise ValueError("No non-zero data could be prepared for plotting")
    
    pubchem_features = plot_df['Feature'].unique().tolist()
    
    if top_n is not None:
        feature_totals = plot_df.groupby('Feature')['Value'].sum()
        pubchem_features = feature_totals.nlargest(top_n).index.tolist()
    
    plot_df = plot_df[plot_df['Feature'].isin(pubchem_features)]
    
    # Create subplots
    n_features = len(pubchem_features)
    n_cols = min(3, n_features)
    n_rows = ceil(n_features / n_cols)
    
    fig, axs = plt.subplots(n_rows, n_cols, 
                           figsize=(figsize_per_plot[0] * n_cols, 
                                   figsize_per_plot[1] * n_rows),
                           squeeze=False)
    axs = axs.flatten()
    
        # Plot each feature
    for i, feature in enumerate(pubchem_features):
        ax = axs[i]
        feature_data = plot_df[plot_df['Feature'] == feature]
        
        # Boxplot
        sns.boxplot(
            data=feature_data,
            x='Condition',
            y='Value',
            hue='Condition',
            ax=ax,
            width=0.5,
            linewidth=1.5,
            legend=False,
            palette='Set2'
        )
        
        # Strip plot for individual points
        sns.stripplot(
            data=feature_data,
            x='Condition',
            y='Value',
            ax=ax,
            color='black',
            alpha=0.05,
            jitter=True,
            size=1.5,
            zorder=0,
            edgecolor='none'
        )
        
        # Style
        wrapped_label = "\n".join(textwrap.wrap(feature.replace('_', ' '), width=20))
        ax.set_title(wrapped_label, fontsize=14, pad=15)
        ax.set_xlabel('')
        ax.set_ylabel('log(1 + feature_value × abundance)', fontsize=14)
        ax.tick_params(axis='x', rotation=0)
        
        # Remove margins so we have full control
        ax.margins(y=0) 
        
        # Determine the visual limit (Added 25% buffer instead of 15%)
        y_max = feature_data['Value'].max()
        y_limit = y_max * 1.25 if y_max > 0 else 1.0
        
        # Set the explicit limit of the Y-axis
        ax.set_ylim(0, y_limit)
        
        # Statistics and Annotation
        if add_stats:
            primary_vals = feature_data[feature_data['Condition'] == 'Primary']['Value']
            met_vals = feature_data[feature_data['Condition'] == 'Metastasis']['Value']
            
            if len(primary_vals) > 0 and len(met_vals) > 0:
                try:
                    stat, pval = mannwhitneyu(primary_vals, met_vals, alternative='two-sided')
                    
                    if pval < 0.001: sig_text = '***'
                    elif pval < 0.01: sig_text = '**'
                    elif pval < 0.05: sig_text = '*'
                    else: sig_text = 'ns'
                    
                    # Anchor the bracket and text into the 25% buffer space
                    y_line = y_limit * 0.88  # Brackets sit slightly lower now
                    y_text = y_limit * 0.96  # Text sits above the bracket
                    
                    if pval < 0.05:
                        ax.plot([0, 0, 1, 1], [y_line, y_line * 1.02, y_line * 1.02, y_line], lw=1.5, c='k')
                        ax.text(0.5, y_text, f'{sig_text} (p={pval:.3f})', 
                               ha='center', fontsize=10, fontweight='bold')
                    else:
                        ax.text(0.5, y_line, f'{sig_text}\np={pval:.3f}', 
                               ha='center', fontsize=10, fontweight='bold')
                except Exception:
                    pass
        # ---------------- FIX END ----------------
    
    # Hide empty subplots
    for i in range(n_features, len(axs)):
        axs[i].set_visible(False)
    
    plt.tight_layout(pad=3.0, h_pad=4.0, w_pad=3.0)
    
    if show_plot:
        plt.show()
    
    return fig, axs[:n_features], plot_df, feature_sample_size

# Usage:
fig, axs, plot_df, feature_sample_size = plot_pubchem_boxplots(
    unified_features=high_conf_top_features,
    features_filtered=features_filtered,
    ids=ids,
    metadata_col='condition',
    figsize_per_plot=(5, 6),
    add_stats=True
)
print(f"Minimum sample size per feature and condition: {feature_sample_size}")


### Confident vs. underconfident
- can the score of JESTR be used as a novelity score
- if all features are less in underconfident predictions

In [ ]:
print("high confidence top features:")
print(high_conf_unified_features['0.55_487.9635m/z'].keys())
print("\nunified features:")
print(pubchem_unified_features['0.55_487.9635m/z'].keys())  

In [ ]:
print(len(pubchem_unified_features))

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import math
from scipy.stats import pearsonr

# --- Extract rows from your dictionary ---
rows = []
for feature_key, feature_data in pubchem_unified_features.items():
    score = feature_data["score"]
    pubchem = feature_data["pubchem"]
    row = {"feature": feature_key, "score": score}
    row.update(pubchem)
    rows.append(row)

df = pd.DataFrame(rows)

# --- PubChem columns ---
pubchem_cols = [
    "PubMed_Count", "Patent_Count", "DrugMedicInfo", "FoodRelated",
    "PharmacoInfo", "SafetyInfo", "ToxicityInfo", "KnownUse", "DisorderDisease"
]

# --- Compute total across ALL PubChem features ---
df["PubChem_Total"] = df[pubchem_cols].sum(axis=1)

def true_percentile(series):
    values = series.values
    N = len(values)
    return pd.Series([(values < v).sum() / N for v in values], index=series.index)


# Compute percentile ranks for each PubChem feature
pct_df = df[pubchem_cols].apply(true_percentile)

# For each datapoint, average only over features that are non-zero
pct_sums = []
for idx, row in df[pubchem_cols].iterrows():
    nonzero_features = pubchem_cols

    if len(nonzero_features) == 0:
        pct_sums.append(0)
        continue
    pct_values = pct_df.loc[idx, nonzero_features]
    pct_sums.append(pct_values.mean())

df["PubChem_Total_Percentile"] = pct_sums


main_cols = ["Patent_Count", "DisorderDisease", "PubChem_Total_Percentile"]

n = len(main_cols)
cols = 1
rows = n

fig, axes = plt.subplots(rows, cols, figsize=(10, 6 * rows))
axes = axes.flatten()

# Global style adjustments
plt.rcParams.update({
    "font.size": 14,          # base font size
    "axes.labelsize": 16,     # axis labels
    "axes.titlesize": 18,     # titles
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 14,
    "lines.linewidth": 3      # thicker default lines
})

for ax, col in zip(axes, main_cols):

    df_nonzero = df[df[col] != 0]
    if len(df_nonzero) < 3:
        ax.set_title(f"{col} (not enough non-zero values)")
        ax.axis("off")
        continue

    r, p = pearsonr(df_nonzero["score"], df_nonzero[col])

    sns.regplot(
        data=df_nonzero,
        x="score",
        y=col,
        ax=ax,
        scatter_kws={"alpha": 0.5, "s": 60},   # larger points
        line_kws={"color": "red", "linewidth": 3},  # thicker regression line
        ci=None
    )

    ax.set_title(f"{col}\nr = {r:.3f}, p = {p:.3e}", pad=14)
    ax.set_xlabel("Similarity Score")
    ax.set_ylabel(col)

    ax.grid(True, linestyle="--", alpha=0.4)  # subtle grid for readability

plt.tight_layout()
plt.show()



In [ ]:
appendix_cols = [
    "PubMed_Count", "DrugMedicInfo", "FoodRelated",
    "PharmacoInfo", "SafetyInfo", "ToxicityInfo", "KnownUse"
]

n = len(appendix_cols)
cols = 2
rows = math.ceil(n / cols)

# --- FONT SETTINGS MUST COME FIRST ---
plt.rcParams.update({
    "font.size": 22,
    "axes.labelsize": 24,
    "axes.titlesize": 28,
    "xtick.labelsize": 22,
    "ytick.labelsize": 22,
})

# --- Force Seaborn to respect these settings ---
sns.set_theme(style="whitegrid", rc={
    "axes.labelsize": 24,
    "axes.titlesize": 28,
    "xtick.labelsize": 22,
    "ytick.labelsize": 22,
})

fig, axes = plt.subplots(rows, cols, figsize=(22, 6 * rows))
axes = axes.flatten()

# --- Increase spacing between graphs ---
fig.subplots_adjust(hspace=0.55, wspace=0.35)

for ax, col in zip(axes, appendix_cols):

    df_nonzero = df[df[col] != 0]
    if len(df_nonzero) < 3:
        ax.set_title(f"{col} (not enough non-zero values)")
        ax.axis("off")
        continue

    r, p = pearsonr(df_nonzero["score"], df_nonzero[col])

    sns.regplot(
        data=df_nonzero,
        x="score",
        y=col,
        ax=ax,
        scatter_kws={"alpha": 0.5, "s": 60},
        line_kws={"color": "red", "linewidth": 3},
        ci=None
    )

    ax.set_title(f"{col}\nr = {r:.3f}, p = {p:.3e}", pad=14)
    ax.set_xlabel("Similarity Score")
    ax.set_ylabel(col)

    ax.grid(True, linestyle="--", alpha=0.4)

# Remove unused axes
for ax in axes[len(appendix_cols):]:
    ax.remove()

plt.tight_layout()
plt.show()

